# Fish2 ROI → soma-in-ROI → morphology cluster explorer

Student/Google Colab notebook for the Fish2 EM dataset.

This notebook:
1. Connects to Fish2 neuPrint.
2. Lists all ROIs.
3. Lets the user change `ROI_QUERY` (default: `Raphe_Superior`).
4. Fetches neurons touching matching ROIs.
5. Restricts to neurons whose soma coordinates are inside the chosen ROI selection:
   - preferably via a precomputed membership CSV from the local analysis;
   - otherwise via the same coordinate-based fallback proxy.
6. Plots selected soma locations in 3D.
7. Plots selected skeletons.
8. Loads saved morphology assignments and accesses cluster 7.
9. Optionally shows cluster 7 on a saved UMAP.

## Token setup for this teaching notebook

This version intentionally uses a simple token variable in the configuration cell:

```python
NEUPRINT_TOKEN = "PASTE_NEUPRINT_TOKEN_HERE"
```

Paste the token manually before running the connection cell.

**Never push a real token to a public GitHub repository.** Before committing the notebook, restore the placeholder string.


In [ ]:
import sys, subprocess, importlib.util
required = {
    "neuprint":"neuprint-python","pandas":"pandas","numpy":"numpy",
    "plotly":"plotly","tqdm":"tqdm","sklearn":"scikit-learn"
}
missing=[p for m,p in required.items() if importlib.util.find_spec(m) is None]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install","-q",*missing])
print("Dependencies ready.")

In [ ]:
from pathlib import Path
import os, subprocess

REPO_URL="https://github.com/mduram/fish2-raphe-morphology.git"
REPO_DIR=Path("/content/fish2-raphe-morphology")

try:
    import google.colab
    IN_COLAB=True
except ImportError:
    IN_COLAB=False

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git","clone","--depth","1",REPO_URL,str(REPO_DIR)],check=True)
    os.chdir(REPO_DIR)

print("Working directory:", Path.cwd())

In [ ]:
from pathlib import Path
from getpass import getpass
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from neuprint import Client, fetch_neurons, fetch_synapses, NeuronCriteria as NC, SynapseCriteria as SC
from tqdm.auto import tqdm
from IPython.display import display
from sklearn.covariance import MinCovDet

NEUPRINT_SERVER="neuprint-fish2.janelia.org"
NEUPRINT_DATASET="fish2"

# ------------------------------------------------------------
# MANUAL TOKEN ENTRY FOR COLAB / TEACHING USE
# ------------------------------------------------------------
# Paste the neuPrint token between the quotes before running.
# WARNING: do not commit a real token to a public GitHub repo.
NEUPRINT_TOKEN = "PASTE_NEUPRINT_TOKEN_HERE"


ROI_QUERY="Raphe_Superior"  # students can change this
PRECOMPUTED_SOMA_IN_ROI_CSV=Path("data/raphe_superior_soma_in_roi_bodyIds.csv")
CLUSTER_ASSIGNMENTS_CSV=Path("data/selected_raphe_active_assignments.csv")
ACTIVE_UMAP_CSV=Path("data/selected_raphe_active_umap.csv")
BRAIN_SHELL_NPZ=Path("data/fish2_clean_brain_shell.npz")

CLUSTER_ID_TO_SHOW=7
MAX_SOMA_IN_ROI_SKELETONS_TO_PLOT=120
MAX_CLUSTER_SKELETONS_TO_PLOT=None

CACHE_DIR=Path("student_cache")
SKEL_DIR=CACHE_DIR/"skeletons"
SKEL_DIR.mkdir(parents=True,exist_ok=True)

In [ ]:
# ============================================================
# Connect to Fish2 neuPrint
# ============================================================
# Before running this cell, paste the token into:
#
#     NEUPRINT_TOKEN = "PASTE_NEUPRINT_TOKEN_HERE"
#
# in the configuration cell above.
#
# IMPORTANT:
# - This is convenient for a teaching Colab notebook.
# - Do NOT commit a real token to a public GitHub repository.
# - Before git push, restore the placeholder string.

token = str(NEUPRINT_TOKEN).strip()

if (
    not token
    or token == "PASTE_NEUPRINT_TOKEN_HERE"
    or "PASTE_NEUPRINT_TOKEN_HERE" in token
):
    raise RuntimeError(
        "Please paste a valid neuPrint token into the NEUPRINT_TOKEN "
        "variable in the configuration cell above."
    )

c = Client(
    NEUPRINT_SERVER,
    dataset=NEUPRINT_DATASET,
    token=token,
)

print("Connected:", c.server, c.dataset)


In [ ]:
all_rois=sorted(c.meta["roiInfo"].keys())
roi_df=pd.DataFrame({"roi":all_rois})
roi_df["matches_query"]=roi_df["roi"].str.contains(ROI_QUERY,case=False,regex=False)
display(roi_df)

selected_rois=[r for r in all_rois if ROI_QUERY.lower() in r.lower()]
print("Selected ROIs:")
for r in selected_rois:
    print(" ",r)
if not selected_rois:
    raise RuntimeError(f"No ROI matched {ROI_QUERY!r}")

In [ ]:
touching_neurons,_=fetch_neurons(NC(rois=selected_rois,roi_req="any"),client=c)
touching_neurons=(touching_neurons.copy()
                  .drop_duplicates("bodyId")
                  .sort_values("bodyId")
                  .reset_index(drop=True))
touching_neurons["bodyId"]=touching_neurons["bodyId"].astype(int)
print("Touching neurons:", len(touching_neurons))
display(touching_neurons.head(20))

In [ ]:
def extract_xyz(v):
    if v is None: return None
    if isinstance(v,dict) and all(k in v for k in ("x","y","z")):
        return np.array([v["x"],v["y"],v["z"]],float)
    if hasattr(v,"x") and hasattr(v,"y") and hasattr(v,"z"):
        return np.array([v.x,v.y,v.z],float)
    if isinstance(v,(list,tuple,np.ndarray)) and len(v)>=3:
        return np.array(v[:3],float)
    return None

if "somaLocation" not in touching_neurons.columns:
    raise RuntimeError("No somaLocation column in neuron table.")

rows=[]
for r in touching_neurons.itertuples():
    xyz=extract_xyz(getattr(r,"somaLocation",None))
    if xyz is not None and np.isfinite(xyz).all():
        rows.append({"bodyId":int(r.bodyId),"x":xyz[0],"y":xyz[1],"z":xyz[2]})
touching_somas=pd.DataFrame(rows)
print("Soma coords:",len(touching_somas),"/",len(touching_neurons))
display(touching_somas.head())

## Soma-in-ROI filtering

The notebook first looks for a precomputed local membership CSV, which is the recommended reproducible route.

If absent, it falls back to a robust spatial proxy built from synapse coordinates inside the chosen ROI(s). That fallback is not an exact ROI-mesh point-in-volume test.

In [ ]:
def fetch_roi_synapse_cloud(body_ids,rois,batch_size=300):
    chunks=[]
    ids=[int(x) for x in body_ids]
    for start in tqdm(range(0,len(ids),batch_size),desc="In-ROI synapses"):
        batch=ids[start:start+batch_size]
        syn=fetch_synapses(
            NC(bodyId=batch),
            SC(rois=rois,primary_only=False),
            client=c
        )
        if len(syn): chunks.append(syn.copy())
    if not chunks:
        return pd.DataFrame(columns=["x","y","z"])
    syn=pd.concat(chunks,ignore_index=True)
    if {"x","y","z"}.issubset(syn.columns):
        pts=syn[["x","y","z"]].astype(float).copy()
    elif "location" in syn.columns:
        arr=[]
        for v in syn["location"]:
            p=extract_xyz(v)
            arr.append([np.nan,np.nan,np.nan] if p is None else p.tolist())
        pts=pd.DataFrame(arr,columns=["x","y","z"])
    else:
        raise RuntimeError(f"Cannot find xyz in synapse table: {list(syn.columns)}")
    return pts.dropna().reset_index(drop=True)

touching_ids=touching_neurons["bodyId"].astype(int).tolist()

if PRECOMPUTED_SOMA_IN_ROI_CSV.exists():
    pre=pd.read_csv(PRECOMPUTED_SOMA_IN_ROI_CSV)
    pre["bodyId"]=pre["bodyId"].astype(int)
    soma_in_roi=touching_somas.merge(pre[["bodyId"]].drop_duplicates(),on="bodyId",how="inner")
    soma_filter_method="precomputed_local_membership"
else:
    pts=fetch_roi_synapse_cloud(touching_ids,selected_rois)
    X=pts[["x","y","z"]].to_numpy(float)
    robust=MinCovDet(random_state=0).fit(X)
    d2_syn=robust.mahalanobis(X)
    cutoff=float(np.quantile(d2_syn,0.975))
    S=touching_somas[["x","y","z"]].to_numpy(float)
    d2_soma=robust.mahalanobis(S)
    soma_in_roi=touching_somas.loc[d2_soma<=cutoff].copy()
    soma_filter_method="robust_synapse_cloud_proxy"

soma_in_roi_ids=sorted(soma_in_roi["bodyId"].astype(int).unique().tolist())
print("Method:",soma_filter_method)
print("Selected soma-in-ROI neurons:",len(soma_in_roi_ids))
display(soma_in_roi.head(20))
print(soma_in_roi_ids)

In [ ]:
fig=go.Figure()

if BRAIN_SHELL_NPZ.exists():
    shell=np.load(BRAIN_SHELL_NPZ)
    v=np.asarray(shell["vertices"],float)
    f=np.asarray(shell["faces"],int)
    fig.add_trace(go.Mesh3d(
        x=v[:,0],y=v[:,1],z=v[:,2],
        i=f[:,0],j=f[:,1],k=f[:,2],
        color="rgb(185,190,195)",opacity=0.05,
        flatshading=True,name="Brain shell",showscale=False
    ))

fig.add_trace(go.Scatter3d(
    x=soma_in_roi["x"],y=soma_in_roi["y"],z=soma_in_roi["z"],
    mode="markers",
    marker=dict(size=4,color="gold",opacity=0.9),
    text=soma_in_roi["bodyId"].astype(str),
    hovertemplate="bodyId=%{text}<extra></extra>",
    name=f"Soma-in-ROI (n={len(soma_in_roi)})"
))

fig.update_layout(
    title=f"Soma locations selected for {ROI_QUERY}",
    scene=dict(xaxis=dict(visible=False),yaxis=dict(visible=False),zaxis=dict(visible=False),
               aspectmode="data",bgcolor="white"),
    paper_bgcolor="white",height=800,margin=dict(l=0,r=0,t=60,b=0)
)
fig.show()

In [ ]:
def fetch_or_load_skeleton(body_id,heal=True):
    body_id=int(body_id)
    path=SKEL_DIR/f"{body_id}.swc"
    if not path.exists():
        try:
            c.fetch_skeleton(body_id,heal=heal,export_path=str(path))
        except Exception as e:
            print("Failed",body_id,e); return None
    rows=[]
    with open(path,"r",errors="ignore") as fh:
        for line in fh:
            line=line.strip()
            if not line or line.startswith("#"): continue
            p=line.split()
            if len(p)<7: continue
            try:
                rows.append([int(float(p[0])),int(float(p[1])),float(p[2]),float(p[3]),
                             float(p[4]),float(p[5]),int(float(p[6]))])
            except ValueError:
                pass
    if not rows: return None
    return pd.DataFrame(rows,columns=["node_id","type","x","y","z","radius","parent_id"])

def swc_edges(swc):
    xyz={int(r.node_id):(float(r.x),float(r.y),float(r.z)) for r in swc.itertuples()}
    xs=[]; ys=[]; zs=[]
    for r in swc.itertuples():
        pid=int(r.parent_id); nid=int(r.node_id)
        if pid<0 or pid not in xyz: continue
        x1,y1,z1=xyz[pid]; x2,y2,z2=xyz[nid]
        xs += [x1,x2,None]; ys += [y1,y2,None]; zs += [z1,z2,None]
    return xs,ys,zs

def plot_skeleton_set(body_ids,title,color,max_neurons=None,width=1.4):
    ids=[int(x) for x in body_ids]
    if max_neurons is not None: ids=ids[:int(max_neurons)]
    fig=go.Figure(); plotted=[]
    for bid in tqdm(ids,desc="Skeletons"):
        swc=fetch_or_load_skeleton(bid)
        if swc is None: continue
        xs,ys,zs=swc_edges(swc)
        fig.add_trace(go.Scatter3d(
            x=xs,y=ys,z=zs,mode="lines",
            line=dict(color=color,width=width),
            name=str(bid),hoverinfo="name",showlegend=False
        ))
        plotted.append(bid)
    fig.update_layout(
        title=f"{title}<br><sup>n={len(plotted)}</sup>",
        scene=dict(xaxis=dict(visible=False),yaxis=dict(visible=False),zaxis=dict(visible=False),
                   aspectmode="data",bgcolor="white"),
        paper_bgcolor="white",height=800,margin=dict(l=0,r=0,t=70,b=0)
    )
    fig.show()
    return fig,plotted

In [ ]:
soma_in_roi_skeleton_fig,soma_in_roi_skeleton_ids=plot_skeleton_set(
    soma_in_roi_ids,
    title=f"Soma-in-ROI neurons for {ROI_QUERY}",
    color="rgb(70,130,220)",
    max_neurons=MAX_SOMA_IN_ROI_SKELETONS_TO_PLOT,
    width=1.2
)

In [ ]:
if not CLUSTER_ASSIGNMENTS_CSV.exists():
    raise FileNotFoundError(
        f"Missing {CLUSTER_ASSIGNMENTS_CSV}. Add the chosen active assignment CSV to data/."
    )

assignments=pd.read_csv(CLUSTER_ASSIGNMENTS_CSV)
assignments["bodyId"]=assignments["bodyId"].astype(int)
assignments["cluster"]=assignments["cluster"].astype(int)

cluster7=assignments.loc[assignments["cluster"]==int(CLUSTER_ID_TO_SHOW)].copy().sort_values("bodyId")
cluster7_ids=cluster7["bodyId"].astype(int).tolist()

print("Available clusters:",sorted(assignments["cluster"].unique().tolist()))
print(f"Cluster {CLUSTER_ID_TO_SHOW}: n={len(cluster7_ids)}")
display(cluster7)
print(cluster7_ids)

In [ ]:
cluster7_fig,cluster7_plotted_ids=plot_skeleton_set(
    cluster7_ids,
    title=f"Selected-Raphe morphology cluster {CLUSTER_ID_TO_SHOW}",
    color="rgb(235,140,35)",
    max_neurons=MAX_CLUSTER_SKELETONS_TO_PLOT,
    width=2.0
)

In [ ]:
if ACTIVE_UMAP_CSV.exists():
    emb=pd.read_csv(ACTIVE_UMAP_CSV)
    emb["bodyId"]=emb["bodyId"].astype(int)

    if {"umap1","umap2"}.issubset(emb.columns):
        xcol,ycol="umap1","umap2"
    elif {"UMAP1","UMAP2"}.issubset(emb.columns):
        xcol,ycol="UMAP1","UMAP2"
    else:
        raise RuntimeError(f"UMAP CSV lacks expected columns: {list(emb.columns)}")

    plot_df=emb.merge(assignments[["bodyId","cluster"]],on="bodyId",how="left")
    fig=go.Figure()
    fig.add_trace(go.Scatter(
        x=plot_df[xcol],y=plot_df[ycol],mode="markers",
        marker=dict(size=7,color="lightgray",opacity=0.65),
        text=plot_df["bodyId"].astype(str),name="All cells"
    ))
    hit=plot_df.loc[plot_df["bodyId"].isin(cluster7_ids)]
    fig.add_trace(go.Scatter(
        x=hit[xcol],y=hit[ycol],mode="markers",
        marker=dict(size=11,color="orange",line=dict(width=1,color="black")),
        text=hit["bodyId"].astype(str),name=f"Cluster {CLUSTER_ID_TO_SHOW}"
    ))
    fig.update_layout(title=f"Cluster {CLUSTER_ID_TO_SHOW} on active UMAP",
                      xaxis_title=xcol,yaxis_title=ycol,
                      paper_bgcolor="white",plot_bgcolor="white",height=700)
    fig.show()
else:
    print("Optional UMAP file not found:",ACTIVE_UMAP_CSV)

## Repository files expected

For a one-click reproducible Colab experience, add the following small local-analysis outputs under `data/`:

- `raphe_superior_soma_in_roi_bodyIds.csv`
- `selected_raphe_active_assignments.csv`
- `selected_raphe_active_umap.csv`
- `fish2_clean_brain_shell.npz`

Skeletons can be fetched live from neuPrint, so SWC files do not need to be committed.